# Convolutional Neural Networks

Pichhli unit mein humne seekha ki multi-layered neural network ko class definition se kaise define kiya ja sakta hai, lekin woh networks general the aur computer vision tasks ke liye specially design nahi kiye gaye the. Is unit mein hum **Convolutional Neural Networks (CNNs)** ke baare mein jaanenge, jo khaas taur par computer vision ke liye design kiye gaye hain.

Computer vision general classification se alag hai, kyunki jab hum kisi picture mein ek certain object dhoondhne ki koshish karte hain, toh hum image ko scan karte hain aur kuch khaas **patterns** aur unke combinations dhoondhte hain. Jaise, jab hum billi dhoondh rahe hote hain, toh pehle horizontal lines dekh sakte hain, jo whiskers bana sakti hain, aur phir whiskers ke ek certain combination se humein pata chal sakta hai ki yeh sach mein billi ki picture hai. Kuch patterns ki relative position aur presence important hoti hai, na ki image par unki exact position.

Patterns extract karne ke liye, hum **convolutional filters** ka concept use karenge. Lekin sabse pehle, chalo woh saari dependencies aur functions load karte hain, jo humne pichhli units mein define kiye the.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt
from torchinfo import summary
import numpy as np

from pytorchcv import load_mnist, train, plot_results, plot_convolution, display_dataset
load_mnist(batch_size=128)

## Convolutional filters

Convolutional filters chhoti windows hoti hain jo image ke har pixel par chalti hain aur aas-paas ke pixels ka weighted average nikalti hain.

Inhe weight coefficients ke matrix se define kiya jaata hai. Chalo apne MNIST handwritten digits par do alag-alag convolutional filters lagane ke examples dekhte hain:

In [ ]:
plot_convolution(torch.tensor([[-1.,0.,1.],[-1.,0.,1.],[-1.,0.,1.]]),'Vertical edge filter')
plot_convolution(torch.tensor([[-1.,-1.,-1.],[0.,0.,0.],[1.,1.,1.]]),'Horizontal edge filter')


Pehla filter **vertical edge filter** kehlata hai, aur ise neeche diye gaye matrix se define kiya gaya hai:
$$
\left(
    \begin{matrix}
     -1 & 0 & 1 \cr
     -1 & 0 & 1 \cr
     -1 & 0 & 1 \cr
    \end{matrix}
\right)
$$
Jab yeh filter ek uniform pixel region par chalta hai, toh saari values jodkar 0 ho jaati hain. Lekin jab yeh image mein kisi vertical edge se takraata hai, toh ek high spike value produce hoti hai. Isi wajah se upar di gayi images mein aap vertical edges ko high aur low values ke roop mein dekh sakte hain, jabki horizontal edges average ho jaate hain.

Iske ulat, jab hum horizontal edge filter lagate hain, toh horizontal lines ubhar kar aati hain, aur vertical lines average ho jaati hain.

Classical computer vision mein, features produce karne ke liye image par kai filters lagaye jaate the, jinhe phir machine learning algorithms classifier banane ke liye use karte the. Lekin, deep learning mein hum aise networks banate hain jo classification problem solve karne ke liye best convolutional filters **khud se** seekhte hain.

Isko karne ke liye, hum **convolutional layers** introduce karte hain.

## Convolutional layers

Convolutional layers `nn.Conv2d` se define ki jaati hain. Humein yeh cheezein specify karni hoti hain:
* `in_channels` - input channels ki sankhya. Hamare case mein hum ek grayscale image ke saath kaam kar rahe hain, isliye input channels ki sankhya 1 hai.
* `out_channels` - use kiye jaane wale filters ki sankhya. Hum 9 alag-alag filters use karenge, jo network ko yeh pata lagane ka kaafi mauka denge ki hamare scenario ke liye kaun se filters sabse achhe hain.
* `kernel_size` - sliding window ka size. Aam taur par 3x3 ya 5x5 filters use kiye jaate hain.

Sabse simple CNN mein ek convolutional layer hogi. 28x28 input size ko dekhte hue, nau 5x5 filters lagane ke baad humare paas 9x24x24 ka ek tensor hoga (spatial size chhota ho jaata hai, kyunki 28 pixels mein 5 length wala sliding interval sirf 24 positions par fit ho sakta hai).

Convolution ke baad, hum 9x24x24 tensor ko ek single vector mein flatten kar dete hain, jiska size 5184 hota hai, aur phir ek linear layer add karte hain, jo 10 classes produce karti hai. Layers ke beech mein hum `relu` activation function bhi use karte hain.

In [ ]:
class OneConv(nn.Module):
    def __init__(self):
        super(OneConv, self).__init__()
        self.conv = nn.Conv2d(in_channels=1,out_channels=9,kernel_size=(5,5))
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(5184,10)

    def forward(self, x):
        x = nn.functional.relu(self.conv(x))
        x = self.flatten(x)
        x = nn.functional.log_softmax(self.fc(x),dim=1)
        return x

net = OneConv()

summary(net,input_size=(1,1,28,28))

Aap dekh sakte hain ki is network mein lagbhag 50k trainable parameters hain, jabki fully connected multi-layered networks mein yeh sankhya lagbhag 80k hoti hai. Isse humein chhote datasets par bhi achhe results paane ki permission milti hai, kyunki convolutional networks generalization mein kaafi behtar hote hain.

In [ ]:
hist = train(net,train_loader,test_loader,epochs=5)
plot_results(hist)

Jaisa ki aap dekh sakte hain, hum pichhli unit ke fully connected networks ke mukable zyada accuracy aur kaafi tez results paa paa rahe hain.

Hum apni trained convolutional layers ke weights ko bhi visualize kar sakte hain, taaki samajhne ki koshish kar sakein ki asal mein kya ho raha hai:

In [ ]:
fig,ax = plt.subplots(1,9)
with torch.no_grad():
    p = next(net.conv.parameters())
    for i,x in enumerate(p):
        ax[i].imshow(x.detach().cpu()[0,...])
        ax[i].axis('off')

Aap dekh sakte hain ki inme se kuch filters tirchi lines pehchaanne mein capable lagte hain, jabki baaki kaafi random dikhte hain.

## Multi-layered CNNs aur pooling layers

Pehli convolutional layer basic patterns dhoondhti hai, jaise horizontal ya vertical lines, lekin hum inke upar aur convolutional layers laga sakte hain taaki higher-level patterns pehchaane ja sakein, jaise basic shapes. Phir aur convolutional layers in shapes ko picture ke kuch hisson mein jod sakti hain, aur aakhir mein us object tak pahunch sakti hain jise hum classify karne ki koshish kar rahe hain.

Aisa karte waqt, hum ek aur trick bhi laga sakte hain: image ka spatial size kam karna. Ek baar jab humne pata laga liya ki sliding 3x3 window ke andar ek horizontal line hai, toh yeh utna important nahi rehta ki woh exactly kis pixel par thi. Isliye hum image ka size "scale down" kar sakte hain, jo **pooling layers** mein se kisi ek ko use karke kiya jaata hai:

 * **Average pooling** ek sliding window leta hai (jaise 2x2 pixels) aur window ke andar ki values ka average nikalta hai.
 * **Max pooling** window ko maximum value se replace kar deta hai. Max pooling ke peeche ka idea yeh hai ki sliding window ke andar ek certain pattern ki presence detect ki jaye.

Is tarah, ek typical CNN mein kai convolutional layers hoti hain, jinke beech mein pooling layers hoti hain taaki image ki dimensions kam ho sakein. Hum filters ki sankhya bhi badhate hain, kyunki jaise-jaise patterns zyada advanced hote jaate hain - humein dekhne ke liye zyada possible interesting combinations milte hain.

![Kai convolutional layers aur pooling layers dikhane wali ek image.](../../../../../lessons/4-ComputerVision/07-ConvNets/images/cnn-pyramid.png)

Spatial dimensions kam karne aur feature/filter dimensions badhane ki wajah se, is architecture ko **pyramid architecture** bhi kehte hain.

In [ ]:
class MultiLayerCNN(nn.Module):
    def __init__(self):
        super(MultiLayerCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(10, 20, 5)
        self.fc = nn.Linear(320,10)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv1(x)))
        x = self.pool(nn.functional.relu(self.conv2(x)))
        x = x.view(-1, 320)
        x = nn.functional.log_softmax(self.fc(x),dim=1)
        return x

net = MultiLayerCNN()
summary(net,input_size=(1,1,28,28))

Is definition ke baare mein kuch baatein dhyan mein rakhiye:
* `Flatten` layer use karne ke bajaye, hum `forward` function ke andar `view` function se tensor ko flatten kar rahe hain. Kyunki flattening layer mein koi trainable weights nahi hote, isliye zaroori nahi hai ki hum apni class ke andar ek alag layer instance banayein.
* Hum apne model mein sirf ek pooling layer use kar rahe hain, kyunki ismein bhi koi trainable parameters nahi hote, aur is ek instance ko effectively dobara use kiya ja sakta hai.
* Trainable parameters ki sankhya (~8.5K) pehle ke cases ke mukable kaafi kam hai. Aisa isliye hota hai kyunki generally convolutional layers mein bahut kam parameters hote hain, aur final dense layer lagane se pehle image ki dimensionality kaafi kam ho jaati hai. Parameters ki kam sankhya ka hamare models par positive effect padta hai, kyunki yeh chhote dataset size par bhi overfitting rokne mein madad karta hai.

In [ ]:
hist = train(net,train_loader,test_loader,epochs=5)

Hum dekh sakte hain ki hum sirf ek layer ke mukable zyada accuracy paa rahe hain, aur woh bhi bahut tezi se - sirf 1 ya 2 epochs mein. Iska matlab hai ki complex network architectures ko yeh samajhne ke liye bahut kam data chahiye ki kya ho raha hai, aur hamari images se general patterns nikaalne ke liye.

## CIFAR-10 dataset se real pictures ke saath experiment karna

Haalaanki hamari handwritten digits recognition ki problem ek simple problem lag sakti hai, ab hum kuch zyada serious karne ke liye taiyaar hain. Chalo alag-alag objects ki pictures ke ek zyada advanced dataset ko explore karte hain, jise [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) kehte hain. Isme 60k 32x32 images hain, jo 10 classes mein baanti gayi hain.

In [ ]:
transform = torchvision.transforms.Compose(
    [torchvision.transforms.ToTensor(),
     torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=14, shuffle=True)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=14, shuffle=False)
classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
display_dataset(trainset,classes=classes)

CIFAR-10 ke liye ek mashhoor architecture [LeNet](https://en.wikipedia.org/wiki/LeNet) kehlata hai, jise *Yann LeCun* ne propose kiya tha. Yeh wahi principles follow karta hai jo humne upar bataye, bas main difference yeh hai ki isme 1 ke bajaye 3 input color channels hote hain.

Hum is model mein ek aur simplification jodte hain - hum output activation function ke roop mein `log_softmax` use nahi karte, aur sirf aakhri fully connected layer ka output return karte hain. Is situation mein, hum model ko optimize karne ke liye `CrossEntropyLoss` loss function use kar sakte hain.

In [ ]:
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.conv3 = nn.Conv2d(16,120,5)
        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(120,64)
        self.fc2 = nn.Linear(64,10)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv1(x)))
        x = self.pool(nn.functional.relu(self.conv2(x)))
        x = nn.functional.relu(self.conv3(x))
        x = self.flat(x)
        x = nn.functional.relu(self.fc1(x))
        x = self.fc2(x)
        return x

net = LeNet()

summary(net,input_size=(1,3,32,32))

In [ ]:
opt = torch.optim.SGD(net.parameters(),lr=0.001,momentum=0.9)
hist = train(net, trainloader, testloader, epochs=3, optimizer=opt, loss_fn=nn.CrossEntropyLoss())

3 epochs ki training ke saath humne jo accuracy hasil ki hai, woh bahut achhi nahi lagti. Lekin, yaad rakhiye ki bina kisi information ke andaza lagane par humein sirf 10% accuracy hi milegi, aur hamari problem asal mein MNIST digit classification ke mukable kaafi zyada mushkil hai. Itne kam training time mein 50% se zyada accuracy paana ek achhi achievement lagti hai.

## Main takeaways

Is unit mein, humne computer vision neural networks ke peeche ka main principle seekha - convolutional networks. Real-life architectures, jo image classification, object detection, aur yahan tak ki image generation networks ko power karte hain, sab CNNs par based hain, bas inme zyada layers aur kuch extra training techniques hoti hain.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->